<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Modelo de lenguaje con tokenización por caracteres

### Consigna
- Seleccionar un corpus de texto sobre el cual entrenar el modelo de lenguaje.
- Realizar el pre-procesamiento adecuado para tokenizar el corpus, estructurar el dataset y separar entre datos de entrenamiento y validación.
- Proponer arquitecturas de redes neuronales basadas en unidades recurrentes para implementar un modelo de lenguaje.
- Con el o los modelos que consideren adecuados, generar nuevas secuencias a partir de secuencias de contexto con las estrategias de greedy search y beam search determístico y estocástico. En este último caso observar el efecto de la temperatura en la generación de secuencias.


### Sugerencias
- Durante el entrenamiento, guiarse por el descenso de la perplejidad en los datos de validación para finalizar el entrenamiento. Para ello se provee un callback.
- Explorar utilizar SimpleRNN (celda de Elman), LSTM y GRU.
- rmsprop es el optimizador recomendado para la buena convergencia. No obstante se pueden explorar otros.


In [1]:
import random
import io
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import keras
from keras import layers
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding, Dropout
from keras.losses import SparseCategoricalCrossentropy

### Datos
Para entrenar el modelo, utilizaremos como conjunto de datos el libro Las mil y una noches. A diferencia de La vuelta al mundo en 80 días, este libro tiene un volumen significativamente mayor de contenido. Por ello, será necesario emplear un enfoque diferente para generar los conjuntos de entrenamiento y validación, optimizando el uso de la memoria RAM.

In [2]:
# descargar de textos.info
import urllib.request

# Para leer y parsear el texto en HTML de wikipedia
import bs4 as bs

In [14]:
# raw_html = urllib.request.urlopen('https://www.textos.info/julio-verne/la-vuelta-al-mundo-en-80-dias/ebook')
raw_html = urllib.request.urlopen('https://www.textos.info/anonimo/las-mil-y-una-noches/ebook')
raw_html = raw_html.read()

# Parsear artículo, 'lxml' es el parser a utilizar
article_html = bs.BeautifulSoup(raw_html, 'lxml')

# Encontrar todos los párrafos del HTML (bajo el tag <p>)
# y tenerlos disponible como lista
article_paragraphs = article_html.find_all('p')

article_text = ''

for para in article_paragraphs:
    article_text += para.text + ' '

# pasar todo el texto a minúscula
article_text = article_text.lower()

In [15]:
# en article text se encuentra el texto de todo el libro
article_text[:1000]

' yo ofrezco\r\n  desnudas, vírgenes,\nintactas y sencillas,\r\n  para mis delicias y el placer de mis\namigos,\r\n  estas noches árabes vividas, soñadas y traducidas sobre su\ntierra natal y sobre el agua\r\n  ellas me fueron dulces durante los ocios\nen remotos mares, bajo un cielo ahora lejano.\r\n  por eso las doy. sencillas, sonrientes y llenas de ingenuidad, como \r\nla musulmana\nschehrazada, su madre suculenta que las dió a luz en el \r\nmisterio;\nfermentando con emoción en los brazos de un príncipe sublime \r\n—lúbrico y\nferoz—, bajo la mirada enternecida de alah, clemente y\n\r\nmisericordioso. al venir al mundo fueron delicadamente mecidas\npor \r\nlas manos de la lustral doniazada, su buena tía, que grabó sus nombres\n\r\nsobre hojas de oro coloreadas de húmedas pedrerías y las cuidó bajo el\n\r\nterciopelo de sus pupilas hasta la adolescencia dura, para esparcirlas\n\r\ndespués, voluptuosas y libres, sobre el mundo oriental, eternizado por\n\r\nsu sonrisa. yo os las entr

In [16]:
len(article_text)

6983238

Dado el tamaño del texto, y con el objetivo de entrenar el modelo en un tiempo razonable, truncaremos el contenido a un máximo de un millón de caracteres.

In [17]:
article_text = article_text[500000:1500000]

### Elegir el tamaño del contexto

En este caso, como el modelo de lenguaje es por caracteres, todo un gran corpus
de texto puede ser considerado un documento en sí mismo y el tamaño de contexto
puede ser elegido con más libertad en comparación a un modelo de lenguaje tokenizado por palabras y dividido en documentos más acotados.

In [18]:
# seleccionamos el tamaño de contexto
max_context_size = 100

In [8]:
# Usaremos las utilidades de procesamiento de textos y secuencias de Keras
from tensorflow.keras.utils import pad_sequences # se utilizará para padding

In [19]:
# en este caso el vocabulario es el conjunto único de caracteres que existe en todo el texto
chars_vocab = set(article_text)


In [20]:
# la longitud de vocabulario de caracteres es:
len(chars_vocab)

70

In [23]:
# Construimos los dicionarios que asignan índices a caracteres y viceversa.
# El diccionario `char2idx` servirá como tokenizador.
char2idx = {k: v for v,k in enumerate(chars_vocab)}
idx2char = {v: k for k,v in char2idx.items()}

In [21]:
char2idx

{'t': 0,
 'b': 1,
 '¡': 2,
 'e': 3,
 'z': 4,
 '.': 5,
 'â': 6,
 ',': 7,
 '\r': 8,
 'p': 9,
 '!': 10,
 'a': 11,
 'í': 12,
 'r': 13,
 'i': 14,
 '"': 15,
 ')': 16,
 'k': 17,
 'q': 18,
 'f': 19,
 ';': 20,
 ' ': 21,
 '«': 22,
 'g': 23,
 '“': 24,
 'ú': 25,
 '3': 26,
 '(': 27,
 'l': 28,
 ':': 29,
 'h': 30,
 '6': 31,
 'é': 32,
 'm': 33,
 '‘': 34,
 'v': 35,
 'c': 36,
 'w': 37,
 'y': 38,
 'j': 39,
 'ó': 40,
 'n': 41,
 '¿': 42,
 'o': 43,
 '\n': 44,
 '\xa0': 45,
 '»': 46,
 's': 47,
 '”': 48,
 '-': 49,
 '?': 50,
 'u': 51,
 '7': 52,
 '`': 53,
 'd': 54,
 '’': 55,
 'á': 56,
 '—': 57,
 "'": 58,
 'ü': 59,
 'x': 60,
 'ñ': 61}

###  Tokenizar

In [24]:
# tokenizamos el texto completo
tokenized_text = [char2idx[ch] for ch in article_text]

In [25]:
tokenized_text[:1000]

[23,
 31,
 13,
 23,
 40,
 13,
 51,
 13,
 23,
 62,
 4,
 31,
 23,
 45,
 13,
 18,
 12,
 0,
 23,
 13,
 0,
 42,
 53,
 51,
 40,
 33,
 13,
 37,
 13,
 63,
 5,
 23,
 4,
 31,
 48,
 9,
 48,
 0,
 57,
 14,
 14,
 4,
 14,
 47,
 23,
 51,
 4,
 23,
 21,
 57,
 4,
 8,
 23,
 42,
 23,
 39,
 47,
 31,
 39,
 12,
 44,
 23,
 13,
 23,
 31,
 47,
 51,
 23,
 10,
 47,
 40,
 47,
 51,
 23,
 37,
 47,
 37,
 4,
 45,
 1,
 47,
 51,
 23,
 40,
 47,
 45,
 23,
 31,
 13,
 51,
 23,
 51,
 4,
 69,
 13,
 51,
 23,
 10,
 4,
 62,
 12,
 62,
 13,
 51,
 8,
 23,
 42,
 48,
 9,
 48,
 37,
 4,
 23,
 62,
 12,
 43,
 47,
 32,
 23,
 38,
 10,
 57,
 4,
 62,
 4,
 51,
 23,
 13,
 10,
 4,
 13,
 14,
 1,
 4,
 63,
 5,
 23,
 4,
 45,
 1,
 47,
 45,
 40,
 4,
 51,
 23,
 4,
 40,
 33,
 36,
 23,
 10,
 12,
 4,
 23,
 13,
 23,
 1,
 12,
 4,
 14,
 14,
 13,
 8,
 23,
 42,
 23,
 31,
 4,
 23,
 62,
 12,
 43,
 4,
 32,
 23,
 38,
 39,
 4,
 48,
 9,
 48,
 13,
 62,
 4,
 31,
 13,
 45,
 1,
 4,
 23,
 10,
 13,
 14,
 13,
 23,
 4,
 45,
 51,
 4,
 69,
 13,
 14,
 37,
 4,
 23,
 4,
 31,
 23

### Organizando y estructurando el dataset

Como mencionamos anteriormente, trabajar con un libro tan extenso genera un alto consumo de memoria RAM, ya que al crear los conjuntos de entrenamiento y validación, se requiere cargar todo el contenido en la RAM simultáneamente. Para resolver este problema, utilizamos la funcionalidad de tf.data.Dataset, que permite cargar y procesar los datos de forma incremental (streaming) o en pequeños lotes. Esto elimina la necesidad de cargar todo el conjunto de datos en la RAM al mismo tiempo, optimizando el uso de los recursos.

In [26]:
import tensorflow as tf


In [27]:
# separaremos el dataset entre entrenamiento y validación.
# `p_val` será la proporción del corpus que se reservará para validación
# `num_val` es la cantidad de secuencias de tamaño `max_context_size` que se usará en validación
p_val = 0.2
num_val = int(np.ceil(len(tokenized_text)*p_val/max_context_size))

In [28]:
train_size = int(len(tokenized_text) * (1-p_val))
train_char_ids = tokenized_text[:train_size]
val_char_ids = tokenized_text[train_size:]

print(f"Caracteres para entrenamiento: {len(train_char_ids)}")
print(f"Caracteres para validación: {len(val_char_ids)}")
print(f"Numero de secuencias de tamaño maximo a utilizar en validación: {num_val}")

Caracteres para entrenamiento: 800000
Caracteres para validación: 200000
Numero de secuencias de tamaño maximo a utilizar en validación: 2000


In [29]:
# Define la longitud de la secuencia de contexto
sequence_length = max_context_size # O el valor que elijas

# Función para crear secuencias input/target
# x es la secuencia de IDs de (sequence_length + 1)
def split_input_target(sequence):
    input_text = sequence[:-1] # Los primeros sequence_length caracteres
    target_text = sequence[1:]  # El último caracter
    return input_text, target_text

# Crear dataset de entrenamiento
train_dataset = tf.data.Dataset.from_tensor_slices(train_char_ids)

# Usar .window() para crear ventanas deslizantes de (sequence_length + 1) caracteres
# shift=1 indica una ventana que se mueve un caracter a la vez (ventana deslizante)
# drop_remainder=True descarta las últimas ventanas incompletas
train_dataset = train_dataset.window(sequence_length + 1, shift=1, drop_remainder=True)

# .flat_map() convierte las "ventanas" (que son Datasets anidados) en un Dataset plano
train_dataset = train_dataset.flat_map(lambda window: window.batch(sequence_length + 1))

# .map() aplica la función split_input_target a cada ventana para obtener (input, target)
train_dataset = train_dataset.map(split_input_target)

# Configurar batching, shuffling y prefetching para eficiencia durante el entrenamiento
BUFFER_SIZE = 10000 # Tamaño del buffer para shuffling (ajústalo según RAM disponible)
BATCH_SIZE = 512    # Tamaño del lote (batch)

train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE) # Optimiza la carga en segundo plano

# Repetir el mismo proceso para el dataset de validación
val_dataset = tf.data.Dataset.from_tensor_slices(val_char_ids)
val_dataset = val_dataset.window(sequence_length + 1, shift=1, drop_remainder=True)
val_dataset = val_dataset.flat_map(lambda window: window.batch(sequence_length + 1))
val_dataset = val_dataset.map(split_input_target)

# NOTA: No suele ser necesario shuffle en el dataset de validación
val_dataset = val_dataset.batch(BATCH_SIZE, drop_remainder=True)
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

print("Datasets creados con tf.data.Dataset")

Datasets creados con tf.data.Dataset


In [ ]:
X = np.array(tokenized_sentences_train[:-1])
y = np.array(tokenized_sentences_train[1:])

Nótese que estamos estructurando el problema de aprendizaje como *many-to-many*:

Entrada: secuencia de tokens [$x_0$, $x_1$, ..., $x_N$]

Target: secuencia de tokens [$x_1$, $x_2$, ..., $x_{N+1}$]

De manera que la red tiene que aprender que su salida deben ser los tokens desplazados en una posición y un nuevo token predicho (el N+1).

La ventaja de estructurar el aprendizaje de esta manera es que para cada token de target se propaga una señal de gradiente por el grafo de cómputo recurrente, que es mejor que estructurar el problema como *many-to-one* en donde sólo una señal de gradiente se propaga.

En este punto tenemos en la variable `tokenized_sentences` los versos tokenizados. Vamos a quedarnos con un conjunto de validación que utilizaremos para medir la calidad de la generación de secuencias con la métrica de Perplejidad.

In [ ]:
X.shape

In [ ]:
X[0,:10]

In [ ]:
y[0,:10]

In [30]:
vocab_size = len(chars_vocab)

# Definiendo el modelo

In [31]:
from keras.layers import Input, TimeDistributed, CategoryEncoding, SimpleRNN, Dense, GRU
from keras.models import Model, Sequential

El modelo que se propone como ejemplo consume los índices de los tokens y los transforma en vectores OHE (en este caso no entrenamos una capa de embedding para caracteres). Esa transformación se logra combinando las capas `CategoryEncoding` que transforma a índices a vectores OHE y `TimeDistributed` que aplica la capa a lo largo de la dimensión "temporal" de la secuencia.

In [32]:
model_original = Sequential()

model_original.add(TimeDistributed(CategoryEncoding(num_tokens=vocab_size, output_mode = "one_hot"),input_shape=(None,1)))
model_original.add(SimpleRNN(200, return_sequences=True, dropout=0.1, recurrent_dropout=0.1 ))
model_original.add(Dense(vocab_size, activation='softmax'))
model_original.compile(loss='sparse_categorical_crossentropy', optimizer='rmsprop')

model_original.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, None, 70)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, None, 200)      │        54,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 70)       │        14,070 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,270 (266.68 KB)

 Trainable params: 68,270 (266.68 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model = Sequential()

model.add(TimeDistributed(CategoryEncoding(num_tokens=vocab_size, output_mode = "one_hot"),input_shape=(None,1)))
model.add(GRU(200, return_sequences=True, dropout=0.1, recurrent_dropout=0.1 ))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='sparse_categorical_crossentropy', optimizer='rmsprop')

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_3              │ (None, None, 68)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, None, 200)      │       162,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, None, 68)       │        13,668 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 175,668 (686.20 KB)

 Trainable params: 175,668 (686.20 KB)

 Non-trainable params: 0 (0.00 B)


### Definir el modelo

Dado que por el momento no hay implementaciones adecuadas de la perplejidad que puedan operar en tiempo de entrenamiento, armaremos un Callback *ad-hoc* que la calcule en cada epoch.

**Nota**: un Callback es una rutina gatillada por algún evento, son muy útiles para relevar datos en diferentes momentos del desarrollo del modelo. En este caso queremos hacer un cálculo cada vez que termina una epoch de entrenamiento.

In [33]:
class PplCallback(keras.callbacks.Callback):

    '''
    Este callback es una solución ad-hoc para calcular al final de cada epoch de
    entrenamiento la métrica de Perplejidad sobre un conjunto de datos de validación.
    La perplejidad es una métrica cuantitativa para evaluar la calidad de la generación de secuencias.
    Además implementa la finalización del entrenamiento (Early Stopping)
    si la perplejidad no mejora después de `patience` epochs.
    '''

    def __init__(self, val_data, history_ppl,patience=5):
      # Guardamos el tf.data.Dataset de validación
        self.val_dataset = val_dataset

        # Lista para guardar el historial de perplejidad (pasada por referencia)
        self.history_ppl = history_ppl

        # Inicializar variables para Early Stopping
        self.min_score = np.inf # Perplejidad mínima vista hasta ahora
        self.patience_counter = 0 # Contador de épocas sin mejora
        self.patience = patience # Número de épocas a esperar antes de detener

        print("PplCallback inicializado con tf.data.Dataset")


    def on_epoch_end(self, epoch, logs=None):

        print(f'\nCalculando perplejidad en la época {epoch + 1}...')

        # Usamos model.evaluate() que es eficiente con tf.data.Dataset
        # Retorna [loss, metric1, metric2, ...]
        # Nuestra pérdida (SparseCategoricalCrossentropy) es el primer elemento.
        # Nota: logs aquí contiene métricas de entrenamiento, no de validación
        # Calcularemos las métricas de validación explícitamente.

        # Desactiva verbosidad de evaluate si no quieres ver el progreso de los batches
        # results = self.model.evaluate(self.val_dataset, verbose=0)
        results = self.model.evaluate(self.val_dataset) # verbose=1 por defecto, muestra progreso

        # La pérdida (SparseCategoricalCrossentropy promediada) es el primer elemento retornado
        val_loss = results

        # Calculamos la perplejidad a partir de la pérdida de validación promediada
        current_score = np.exp(val_loss)

        # Guardamos y mostramos la perplejidad
        self.history_ppl.append(current_score)
        print(f'Época {epoch + 1}: Perplejidad de validación = {current_score:.4f}')

        # Implementación de Early Stopping
        if current_score < self.min_score:
            self.min_score = current_score
            # Opcional: Guardar el modelo con la mejor perplejidad
            self.model.save("best_model_ppl.keras")
            print("¡Mejora detectada! Guardado el modelo con mejor perplejidad.")
            self.patience_counter = 0 # Reiniciar contador si hay mejora
        else:
            self.patience_counter += 1 # Incrementar contador si no hay mejora
            print(f"No hay mejora en la perplejidad. Paciencia: {self.patience_counter}/{self.patience}")
            if self.patience_counter >= self.patience:
                print(f"\nPerplejidad no mejora por {self.patience} épocas. ¡Deteniendo entrenamiento!")
                self.model.stop_training = True # Señal para detener el entrenamiento


### Entrenamiento

In [ ]:
# fiteamos, nótese el agregado del callback con su inicialización. El batch_size lo podemos seleccionar a mano
# en general, lo mejor es escoger el batch más grande posible que minimice el tiempo de cada época.
# En la variable `history_ppl` se guardarán los valores de perplejidad para cada época.
history_ppl_original = []
hist = model_original.fit(train_dataset, epochs=20,validation_data=val_dataset ,callbacks=[PplCallback(val_data=val_dataset, history_ppl=history_ppl_original)])

PplCallback inicializado con tf.data.Dataset
Epoch 1/20
   1560/Unknown 182s 114ms/step - loss: 2.4057

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Calculando perplejidad en la época 1...
390/390 ━━━━━━━━━━━━━━━━━━━━ 39s 100ms/step - loss: 1.8917
Época 1: Perplejidad de validación = 6.6530
¡Mejora detectada! Guardado el modelo con mejor perplejidad.
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 261s 163ms/step - loss: 2.4052 - val_loss: 1.8951
Epoch 2/20
1561/1562 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - loss: 1.9062
Calculando perplejidad en la época 2...
390/390 ━━━━━━━━━━━━━━━━━━━━ 39s 101ms/step - loss: 1.7468
Época 2: Perplejidad de validación = 5.7435
¡Mejora detectada! Guardado el modelo con mejor perplejidad.
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 264s 166ms/step - loss: 1.9061 - val_loss: 1.7481
Epoch 3/20
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - loss: 1.7995
Calculando perplejidad en la época 3...
390/390 ━━━━━━━━━━━━━━━━━━━━ 39s 99ms/step - loss: 1.6943
Época 3: Perplejidad de validación = 5.4514
¡Mejora detectada! Guardado el modelo con mejor perplejidad.
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 260s 165ms/step - loss: 1.7994 - val_loss: 1.6959
Epo

In [ ]:
# fiteamos, nótese el agregado del callback con su inicialización. El batch_size lo podemos seleccionar a mano
# en general, lo mejor es escoger el batch más grande posible que minimice el tiempo de cada época.
# En la variable `history_ppl` se guardarán los valores de perplejidad para cada época.
history_ppl = []
hist = model.fit(train_dataset, epochs=20,validation_data=val_dataset ,callbacks=[PplCallback(val_data=val_dataset, history_ppl=history_ppl)])

PplCallback inicializado con tf.data.Dataset
Epoch 1/20
    624/Unknown 163s 252ms/step - loss: 2.7549

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Calculando perplejidad en la época 1...
155/155 ━━━━━━━━━━━━━━━━━━━━ 26s 168ms/step - loss: 2.0859
Época 1: Perplejidad de validación = 8.1780
¡Mejora detectada! Guardado el modelo con mejor perplejidad.
624/624 ━━━━━━━━━━━━━━━━━━━━ 214s 335ms/step - loss: 2.7544 - val_loss: 2.1014
Epoch 2/20
624/624 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step - loss: 2.1199
Calculando perplejidad en la época 2...
155/155 ━━━━━━━━━━━━━━━━━━━━ 29s 184ms/step - loss: 1.8915
Época 2: Perplejidad de validación = 6.7570
¡Mejora detectada! Guardado el modelo con mejor perplejidad.
624/624 ━━━━━━━━━━━━━━━━━━━━ 327s 441ms/step - loss: 2.1198 - val_loss: 1.9106
Epoch 3/20
624/624 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - loss: 1.9286
Calculando perplejidad en la época 3...
155/155 ━━━━━━━━━━━━━━━━━━━━ 28s 179ms/step - loss: 1.7543
Época 3: Perplejidad de validación = 5.8940
¡Mejora detectada! Guardado el modelo con mejor perplejidad.
624/624 ━━━━━━━━━━━━━━━━━━━━ 321s 439ms/step - loss: 1.9285 - val_loss: 1.7739
Epoch 4/20
5

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Entrenamiento
epoch_count = range(1, len(history_ppl) + 1)
sns.lineplot(x=epoch_count,  y=history_ppl)
plt.show()

In [ ]:
# Cargamos el mejor modelo guardado del entrenamiento para hacer inferencia
model = keras.models.load_model('my_model.keras')


### Predicción del próximo caracter

In [ ]:
# Se puede usar gradio para probar el modelo
# Gradio es una herramienta muy útil para crear interfaces para ensayar modelos
# https://gradio.app/

!pip install -q gradio

In [ ]:
import gradio as gr

def model_response(human_text):

    # Encodeamos
    encoded = [char2idx[ch] for ch in human_text.lower() ]
    # Si tienen distinto largo
    encoded = pad_sequences([encoded], maxlen=max_context_size, padding='pre')

    # Predicción softmax
    y_hat = np.argmax(model.predict(encoded)[0,-1,:])


    # Debemos buscar en el vocabulario el caracter
    # que corresopnde al indice (y_hat) predicho por le modelo
    out_word = ''
    out_word = idx2char[y_hat]

    # Agrego la palabra a la frase predicha
    return human_text + out_word

iface = gr.Interface(
    fn=model_response,
    inputs=["textbox"],
    outputs="text")

iface.launch(debug=True)

### Generación de secuencias

In [ ]:
def generate_seq(model, seed_text, max_length, n_words):
    """
        Exec model sequence prediction

        Args:
            model (keras): modelo entrenado
            seed_text (string): texto de entrada (input_seq)
            max_length (int): máxima longitud de la sequencia de entrada
            n_words (int): números de caracteres a agregar a la sequencia de entrada
        returns:
            output_text (string): sentencia con las "n_words" agregadas
    """
    output_text = seed_text
	# generate a fixed number of words
    for _ in range(n_words):
		# Encodeamos
        encoded = [char2idx[ch] for ch in output_text.lower() ]
		# Si tienen distinto largo
        encoded = pad_sequences([encoded], maxlen=max_length, padding='pre')

		# Predicción softmax
        y_hat = np.argmax(model.predict(encoded,verbose=0)[0,-1,:])
		# Vamos concatenando las predicciones
        out_word = ''

        out_word = idx2char[y_hat]

		# Agrego las palabras a la frase predicha
        output_text += out_word
    return output_text

In [ ]:
input_text='habia una vez'

generate_seq(model, input_text, max_length=max_context_size, n_words=30)

###  Beam search y muestreo aleatorio

In [ ]:
# funcionalidades para hacer encoding y decoding

def encode(text,max_length=max_context_size):

    encoded = [char2idx[ch] for ch in text]
    encoded = pad_sequences([encoded], maxlen=max_length, padding='pre')

    return encoded

def decode(seq):
    return ''.join([idx2char[ch] for ch in seq])

In [ ]:
from scipy.special import softmax

# función que selecciona candidatos para el beam search
def select_candidates(pred,num_beams,vocab_size,history_probs,history_tokens,temp,mode):

  # colectar todas las probabilidades para la siguiente búsqueda
  pred_large = []

  for idx,pp in enumerate(pred):
    pred_large.extend(np.log(pp+1E-10)+history_probs[idx])

  pred_large = np.array(pred_large)

  # criterio de selección
  if mode == 'det':
    idx_select = np.argsort(pred_large)[::-1][:num_beams] # beam search determinista
  elif mode == 'sto':
    idx_select = np.random.choice(np.arange(pred_large.shape[0]), num_beams, p=softmax(pred_large/temp)) # beam search con muestreo aleatorio
  else:
    raise ValueError(f'Wrong selection mode. {mode} was given. det and sto are supported.')

  # traducir a índices de token en el vocabulario
  new_history_tokens = np.concatenate((np.array(history_tokens)[idx_select//vocab_size],
                        np.array([idx_select%vocab_size]).T),
                      axis=1)

  # devolver el producto de las probabilidades (log) y la secuencia de tokens seleccionados
  return pred_large[idx_select.astype(int)], new_history_tokens.astype(int)


def beam_search(model,num_beams,num_words,input,temp=1,mode='det'):

    # first iteration

    # encode
    encoded = encode(input)

    # first prediction
    y_hat = model.predict(encoded,verbose=0)[0,-1,:]

    # get vocabulary size
    vocab_size = y_hat.shape[0]

    # initialize history
    history_probs = [0]*num_beams
    history_tokens = [encoded[0]]*num_beams

    # select num_beams candidates
    history_probs, history_tokens = select_candidates([y_hat],
                                        num_beams,
                                        vocab_size,
                                        history_probs,
                                        history_tokens,
                                        temp,
                                        mode)

    # beam search loop
    for i in range(num_words-1):

      preds = []

      for hist in history_tokens:

        # actualizar secuencia de tokens
        input_update = np.array([hist[i+1:]]).copy()

        # predicción
        y_hat = model.predict(input_update,verbose=0)[0,-1,:]

        preds.append(y_hat)

      history_probs, history_tokens = select_candidates(preds,
                                                        num_beams,
                                                        vocab_size,
                                                        history_probs,
                                                        history_tokens,
                                                        temp,
                                                        mode)

    return history_tokens[:,-(len(input)+num_words):]

In [ ]:
# predicción con beam search
salidas = beam_search(model,num_beams=10,num_words=20,input="habia una vez")

In [ ]:
salidas[0]

In [ ]:
# veamos las salidas
decode(salidas[0])